# TCGA-BRCA: Descarga, Organización e Integración de RNA-seq + Diagnostic Slides

Flujo de trabajo reproducible para obtener datos del **Genomic Data Commons (GDC)** para **TCGA-BRCA**,
organizar los archivos descargados y construir una tabla de correspondencia entre:

- Archivos de expresión génica (RNA-seq)
- Imágenes histopatológicas (**Diagnostic Slides**, formato SVS)

**Archivos requeridos del GDC Portal:**

| Archivo | Origen |
|---|---|
| `gdc_manifest_rnaseq.txt` | Carrito GDC → Manifest (RNA-seq) |
| `gdc_manifest_slides.txt` | Carrito GDC → Manifest (Slides) |
| `metadata.cart.json` | Carrito GDC → Metadata |
| `clinical.cart.json` | Carrito GDC → Clinical |

> **Referencia oficial:** [GDC Data Transfer Tool](https://gdc.cancer.gov/access-data/gdc-data-transfer-tool)


## 1. Estructura de carpetas recomendada

```text
TCGA_BRCA/
├── gdc-client/
│   └── gdc-client.exe          # Windows | gdc-client en Linux/macOS
├── manifest/
│   ├── gdc_manifest_rnaseq.txt
│   └── gdc_manifest_slides.txt
├── metadata/
│   └── metadata.cart.json
├── clinical/
│   └── clinical.cart.json
├── RNAseq/                     # destino de descarga RNA-seq
└── Diagnostic_Slides/          # destino de descarga SVS
```

> La carpeta `gdc-client` puede estar en el PATH del sistema en lugar de dentro del proyecto.


## 2. Imports y configuración global

In [ ]:
from __future__ import annotations

import json
import logging
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd

# ── Logging ──────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

# ── Rutas del proyecto ────────────────────────────────────────────────────────
# Ajusta BASE a la raíz de tu proyecto
BASE = Path(r"C:\TCGA_BRCA")          # Windows
# BASE = Path("/data/TCGA_BRCA")       # Linux / macOS

GDC_CLIENT = BASE / "gdc-client" / "gdc-client.exe"  # ajustar si está en PATH
MANIFEST_RNA   = BASE / "manifest" / "gdc_manifest_rnaseq.txt"
MANIFEST_SLIDE = BASE / "manifest" / "gdc_manifest_slides.txt"
METADATA_PATH  = BASE / "metadata" / "metadata.cart.json"
CLINICAL_PATH  = BASE / "clinical" / "clinical.cart.json"
RNA_DIR        = BASE / "RNAseq"
SLIDES_DIR     = BASE / "Diagnostic_Slides"
OUTPUT_DIR     = BASE / "output"

# ── Mapa completo de tipos de muestra TCGA ───────────────────────────────────
SAMPLE_TYPE_MAP: dict[str, str] = {
    "01": "Primary Solid Tumor",
    "02": "Recurrent Solid Tumor",
    "03": "Primary Blood Derived Cancer - Peripheral Blood",
    "04": "Recurrent Blood Derived Cancer - Bone Marrow",
    "05": "Additional - New Primary",
    "06": "Metastatic",
    "07": "Additional Metastatic",
    "10": "Blood Derived Normal",
    "11": "Solid Tissue Normal",
    "12": "Buccal Cell Normal",
    "13": "EBV Immortalized Normal",
    "14": "Bone Marrow Normal",
    "20": "Cell Lines",
    "40": "Recurrent Blood Derived Cancer - Peripheral Blood",
    "50": "Cell Line Derived Xenograft Tissue",
    "60": "Primary Xenograft Tissue",
    "61": "Cell Line Derived Xenograft Tissue",
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
log.info("Configuración cargada. BASE = %s", BASE)


## 3. Funciones auxiliares

In [ ]:
def parse_patient_id(barcode: str | None, n_blocks: int = 3) -> str | None:
    """
    Extrae el ID de paciente de un barcode TCGA.

    Ejemplo:
        'TCGA-GM-A2DL-01A-11R-A18M-07'  →  'TCGA-GM-A2DL'  (n_blocks=3)
    """
    if not barcode:
        return None
    parts = str(barcode).split("-")
    return "-".join(parts[:n_blocks]) if len(parts) >= n_blocks else None


def parse_sample_code(barcode: str | None) -> str | None:
    """
    Extrae el código de tipo de muestra (posiciones 13-15 del barcode TCGA).

    Ejemplo:
        'TCGA-GM-A2DL-01A-...'  →  '01'
    """
    if not barcode or len(barcode) < 15:
        return None
    return str(barcode)[13:15]


def parse_metadata_entry(item: dict) -> dict:
    """Normaliza un elemento del metadata.cart.json en un diccionario plano."""
    associated = item.get("associated_entities", [])
    entity = associated[0] if associated else {}

    entity_submitter_id = entity.get("entity_submitter_id")
    sample_code = parse_sample_code(entity_submitter_id)

    return {
        "file_id":               item.get("file_id"),
        "file_name":             item.get("file_name"),
        "file_size":             item.get("file_size"),
        "md5sum":                item.get("md5sum"),
        "data_type":             item.get("data_type"),
        "data_format":           item.get("data_format"),
        "data_category":         item.get("data_category"),
        "experimental_strategy": item.get("experimental_strategy"),
        "case_id":               entity.get("case_id"),
        "entity_id":             entity.get("entity_id"),
        "entity_submitter_id":   entity_submitter_id,
        "entity_type":           entity.get("entity_type"),
        "patient_id":            parse_patient_id(entity_submitter_id),
        "sample_code":           sample_code,
        "sample_type":           SAMPLE_TYPE_MAP.get(sample_code, "Unknown") if sample_code else None,
    }


def load_metadata(path: Path) -> pd.DataFrame:
    """Carga y normaliza metadata.cart.json en un DataFrame."""
    if not path.exists():
        raise FileNotFoundError(f"metadata.cart.json no encontrado: {path}")

    with open(path, encoding="utf-8") as fh:
        raw = json.load(fh)

    df = pd.DataFrame([parse_metadata_entry(x) for x in raw])
    log.info("metadata cargado: %d filas, %d columnas", *df.shape)
    return df


def load_clinical(path: Path) -> pd.DataFrame:
    """Carga clinical.cart.json y extrae campos relevantes."""
    if not path.exists():
        log.warning("clinical.cart.json no encontrado: %s", path)
        return pd.DataFrame()

    with open(path, encoding="utf-8") as fh:
        raw = json.load(fh)

    records = []
    for entry in raw:
        demographic = entry.get("demographic", {}) or {}
        diagnoses   = entry.get("diagnoses", [{}]) or [{}]
        dx          = diagnoses[0] if diagnoses else {}
        records.append({
            "patient_id":        parse_patient_id(entry.get("submitter_id")),
            "case_id":           entry.get("case_id"),
            "gender":            demographic.get("gender"),
            "age_at_diagnosis":  dx.get("age_at_diagnosis"),
            "primary_diagnosis": dx.get("primary_diagnosis"),
            "tumor_stage":       dx.get("tumor_stage"),
            "vital_status":      demographic.get("vital_status"),
        })

    df = pd.DataFrame(records)
    log.info("clinical cargado: %d filas", len(df))
    return df


print("✓ Funciones auxiliares definidas")


## 4. Verificar el manifest RNA-seq

In [ ]:
def load_manifest(path: Path) -> pd.DataFrame:
    """Carga un manifest GDC y verifica su formato."""
    if not path.exists():
        raise FileNotFoundError(f"Manifest no encontrado: {path}")

    df = pd.read_csv(path, sep="\t")
    expected_cols = {"id", "filename", "md5", "size", "state"}
    missing = expected_cols - set(df.columns)
    if missing:
        log.warning("Columnas inesperadas en manifest. Faltan: %s", missing)

    log.info("Manifest cargado: %d archivos", len(df))
    return df


manifest_rna = load_manifest(MANIFEST_RNA)
print(f"Dimensiones: {manifest_rna.shape}")
print(f"Columnas:    {manifest_rna.columns.tolist()}")
manifest_rna.head()


## 5. Descargar archivos con `gdc-client`

### Comandos de referencia (ejecutar en terminal)

**RNA-seq (Windows):**
```bat
gdc-client.exe download -m gdc_manifest_rnaseq.txt -d ..\RNAseq
```

**Slides (Windows):**
```bat
gdc-client.exe download -m gdc_manifest_slides.txt -d ..\Diagnostic_Slides
```

**Linux / macOS:**
```bash
gdc-client download -m gdc_manifest_rnaseq.txt -d ../RNAseq
gdc-client download -m gdc_manifest_slides.txt -d ../Diagnostic_Slides
```

> Opciones útiles: `-n <N>` para controlar conexiones paralelas; el DTT reanuda descargas interrumpidas automáticamente.

La celda siguiente lanza la descarga desde Python si el ejecutable existe en la ruta configurada.
Déjala comentada si prefieres ejecutar el DTT manualmente.


In [ ]:
def run_gdc_download(
    manifest: Path,
    dest_dir: Path,
    gdc_client: Path = GDC_CLIENT,
    n_connections: int = 5,
    dry_run: bool = True,
) -> None:
    """
    Lanza gdc-client download desde Python.

    Parameters
    ----------
    dry_run:
        Si True, solo imprime el comando sin ejecutarlo (por defecto).
        Cambia a False para iniciar la descarga real.
    """
    dest_dir.mkdir(parents=True, exist_ok=True)

    # Soporte para gdc-client en PATH (sin ruta absoluta)
    exe = str(gdc_client) if gdc_client.exists() else "gdc-client"

    cmd = [exe, "download", "-m", str(manifest), "-d", str(dest_dir), "-n", str(n_connections)]

    if dry_run:
        print("DRY RUN — comando que se ejecutaría:")
        print(" ".join(cmd))
        return

    log.info("Iniciando descarga: %s → %s", manifest.name, dest_dir)
    result = subprocess.run(cmd, check=False)
    if result.returncode != 0:
        log.error("gdc-client terminó con código %d", result.returncode)
    else:
        log.info("Descarga completada.")


# ── Descarga RNA-seq ──────────────────────────────────────────────────────────
run_gdc_download(MANIFEST_RNA, RNA_DIR, dry_run=True)

# ── Descarga Slides ───────────────────────────────────────────────────────────
# run_gdc_download(MANIFEST_SLIDE, SLIDES_DIR, dry_run=True)


## 6. Cargar y normalizar `metadata.cart.json`

In [ ]:
metadata_df = load_metadata(METADATA_PATH)
print(metadata_df.dtypes)
metadata_df.head(3)


In [ ]:
# Distribución por estrategia experimental y tipo de muestra
print("=== Estrategia experimental ===")
print(metadata_df["experimental_strategy"].value_counts(dropna=False))

print("\n=== Tipo de muestra ===")
print(metadata_df["sample_type"].value_counts(dropna=False))


## 7. Filtrar archivos RNA-seq y anotar tipo de muestra

In [ ]:
rna_seq = metadata_df[
    metadata_df["experimental_strategy"].eq("RNA-Seq")
].copy()

log.info("Archivos RNA-seq: %d | Pacientes únicos: %d",
         len(rna_seq), rna_seq["patient_id"].nunique())

print(f"Archivos RNA-seq : {len(rna_seq):>6,}")
print(f"Pacientes únicos : {rna_seq['patient_id'].nunique():>6,}")
print(f"Data types       : {rna_seq['data_type'].unique().tolist()}")

rna_seq[["patient_id", "entity_submitter_id", "sample_type",
         "data_type", "file_name"]].head(10)


In [ ]:
# Distribución de tipos de muestra en RNA-seq
rna_seq["sample_type"].value_counts(dropna=False).rename("count").to_frame()


## 8. Cargar datos clínicos

In [ ]:
clinical_df = load_clinical(CLINICAL_PATH)

if not clinical_df.empty:
    print(f"Clinical rows : {len(clinical_df)}")
    print(clinical_df.dtypes)
    clinical_df.head(3)
else:
    print("⚠ clinical.cart.json no disponible — se omite.")


## 9. Guardar listado RNA-seq

In [ ]:
COLS_RNA = [
    "patient_id", "entity_submitter_id", "sample_code", "sample_type",
    "file_id", "file_name", "file_size", "md5sum",
    "data_type", "data_format", "experimental_strategy",
    "case_id", "entity_type",
]

# Solo columnas que existan en el DataFrame
cols_present = [c for c in COLS_RNA if c in rna_seq.columns]
listado_rna = rna_seq[cols_present].copy()

out_rna = OUTPUT_DIR / "listado_RNAseq_TCGA_BRCA.tsv"
listado_rna.to_csv(out_rna, sep="\t", index=False)
log.info("Guardado: %s  (%d filas)", out_rna, len(listado_rna))
print(f"✓ {out_rna}")


## 10. Inventariar Diagnostic Slides descargadas

> **Nota importante:** que un paciente tenga RNA-seq **no implica** que exista
> una Diagnostic Slide disponible, y viceversa. La correspondencia debe
> verificarse explícitamente mediante los metadatos del GDC.


In [ ]:
def inventariar_slides(slides_dir: Path) -> pd.DataFrame:
    """
    Recorre el directorio de slides y devuelve un DataFrame con:
    ruta, nombre, tamaño (MB) y el UUID de carpeta (tal como gdc-client lo genera).
    """
    svs_files = sorted(slides_dir.rglob("*.svs"))
    log.info("Archivos SVS encontrados: %d", len(svs_files))

    records = []
    for f in svs_files:
        # gdc-client descarga en subcarpetas cuyo nombre es el file_id (UUID)
        folder_uuid = f.parent.name if f.parent != slides_dir else None
        records.append({
            "svs_path":    f,
            "svs_name":    f.name,
            "folder_uuid": folder_uuid,
            "size_mb":     round(f.stat().st_size / 1_048_576, 2),
        })

    return pd.DataFrame(records)


slides_inv = inventariar_slides(SLIDES_DIR)
print(f"Total SVS : {len(slides_inv)}")
print(f"Tamaño total (GB) : {slides_inv['size_mb'].sum() / 1024:.2f}")
slides_inv.head(10)


## 11. Cruzar slides con metadata del GDC

In [ ]:
# Filtrar metadata de slides (Slide Image / Diagnostic Slide)
slides_meta = metadata_df[
    metadata_df["data_type"].isin(["Slide Image", "Diagnostic Slide"])
    | metadata_df["data_format"].eq("SVS")
].copy()

log.info("Registros de slides en metadata: %d | Pacientes: %d",
         len(slides_meta), slides_meta["patient_id"].nunique())

print(f"Registros slides en metadata : {len(slides_meta):>5,}")
print(f"Pacientes únicos             : {slides_meta['patient_id'].nunique():>5,}")
slides_meta[["patient_id", "file_id", "file_name", "data_type", "sample_type"]].head(10)


## 12. Tabla integrada RNA-seq ↔ Diagnostic Slides

Construcción de la tabla de correspondencia definitiva.


In [ ]:
def build_integrated_table(
    rna_df: pd.DataFrame,
    slides_meta_df: pd.DataFrame,
    clinical_df: pd.DataFrame | None = None,
    how: str = "inner",
) -> pd.DataFrame:
    """
    Cruza RNA-seq con slides a nivel de patient_id.

    Parameters
    ----------
    how : {'inner', 'left', 'outer'}
        - 'inner' : solo pacientes con ambos tipos de archivo (recomendado para modelos multimodales)
        - 'left'  : todos los pacientes RNA-seq, con slides cuando existan
        - 'outer' : todos los pacientes de cualquier fuente
    """
    rna_slim = rna_df[[
        "patient_id", "case_id", "entity_submitter_id", "sample_code",
        "sample_type", "file_id", "file_name", "data_type",
    ]].rename(columns={
        "file_id":   "rna_file_id",
        "file_name": "rna_file_name",
        "data_type": "rna_data_type",
        "case_id":   "rna_case_id",
    })

    slides_slim = slides_meta_df[[
        "patient_id", "file_id", "file_name", "data_type", "sample_type",
    ]].rename(columns={
        "file_id":    "slide_file_id",
        "file_name":  "slide_file_name",
        "data_type":  "slide_data_type",
        "sample_type": "slide_sample_type",
    })

    merged = rna_slim.merge(slides_slim, on="patient_id", how=how)

    # Añadir datos clínicos si están disponibles
    if clinical_df is not None and not clinical_df.empty:
        merged = merged.merge(clinical_df, on="patient_id", how="left")

    log.info("Tabla integrada: %d pares paciente×archivo (%s join)", len(merged), how)
    return merged


integrated = build_integrated_table(
    rna_df=rna_seq,
    slides_meta_df=slides_meta,
    clinical_df=clinical_df if "clinical_df" in dir() and not clinical_df.empty else None,
    how="inner",   # cambiar a 'left' para ver todos los pacientes RNA-seq
)

print(f"Pares RNA-seq ↔ slide : {len(integrated):>5,}")
print(f"Pacientes con ambos   : {integrated['patient_id'].nunique():>5,}")
integrated.head(10)


## 13. Análisis de cobertura

In [ ]:
rna_patients   = set(rna_seq["patient_id"].dropna())
slide_patients = set(slides_meta["patient_id"].dropna())
both_patients  = rna_patients & slide_patients
only_rna       = rna_patients - slide_patients
only_slide     = slide_patients - rna_patients

print(f"Pacientes con RNA-seq únicamente        : {len(only_rna):>5,}")
print(f"Pacientes con slides únicamente          : {len(only_slide):>5,}")
print(f"Pacientes con AMBOS (RNA-seq + slides)   : {len(both_patients):>5,}")
print(f"Total pacientes únicos                   : {len(rna_patients | slide_patients):>5,}")
print(f"\nCobertura slides sobre RNA-seq           : {len(both_patients)/len(rna_patients)*100:.1f} %")


## 14. Verificar integridad de la descarga

Compara los `file_id` del manifest con los UUIDs de las carpetas creadas por `gdc-client`.


In [ ]:
def verificar_descarga(manifest_path: Path, dest_dir: Path) -> pd.DataFrame:
    """
    Compara el manifest con los archivos presentes en disco.
    Devuelve un DataFrame con el estado de cada archivo.
    """
    if not manifest_path.exists():
        log.warning("Manifest no encontrado: %s", manifest_path)
        return pd.DataFrame()

    manifest = pd.read_csv(manifest_path, sep="\t")
    downloaded_uuids = {p.name for p in dest_dir.iterdir() if p.is_dir()} if dest_dir.exists() else set()

    manifest["descargado"] = manifest["id"].isin(downloaded_uuids)
    manifest["estado"] = manifest["descargado"].map({True: "✓ OK", False: "✗ FALTA"})

    n_ok     = manifest["descargado"].sum()
    n_total  = len(manifest)
    log.info("Descarga: %d / %d archivos presentes (%.1f %%)",
             n_ok, n_total, 100 * n_ok / n_total if n_total else 0)

    print(manifest["estado"].value_counts().to_string())
    return manifest[["id", "filename", "size", "descargado", "estado"]]


print("=== RNA-seq ===")
estado_rna = verificar_descarga(MANIFEST_RNA, RNA_DIR)
estado_rna.head(5)


In [ ]:
print("=== Diagnostic Slides ===")
estado_slides = verificar_descarga(MANIFEST_SLIDE, SLIDES_DIR)
estado_slides.head(5)


## 15. Guardar tabla final integrada

In [ ]:
out_integrated = OUTPUT_DIR / "TCGA_BRCA_RNAseq_Slides_integrated.tsv"
integrated.to_csv(out_integrated, sep="\t", index=False)
log.info("Tabla integrada guardada: %s", out_integrated)
print(f"✓ {out_integrated}")
print(f"  Filas    : {len(integrated):,}")
print(f"  Columnas : {len(integrated.columns)}")
print(f"  Columnas : {integrated.columns.tolist()}")


## 16. Consideraciones para modelado multimodal

### ¿Qué etiqueta usar?

Si trabajas **solo con tumor primario** (`sample_code == "01"`), la etiqueta
tumor vs. normal pierde sentido. Opciones recomendadas para TCGA-BRCA:

| Etiqueta | Fuente |
|---|---|
| Subtipo molecular (PAM50): Luminal A, Luminal B, HER2-enriched, Basal-like | [cBioPortal](https://www.cbioportal.org/) → TCGA-BRCA → Clinical Data → `data_clinical_patient.txt` → columna `SUBTYPE` |
| Estado HER2 / ER / PR | Datos clínicos complementarios |
| Supervivencia (OS) | `clinical.cart.json` → `vital_status`, `days_to_death` |

> Los subtipos moleculares **no están disponibles directamente en el GDC Portal**.
> Se obtienen desde [cBioPortal](https://www.cbioportal.org/) descargando los datos clínicos
> del estudio TCGA-BRCA y cruzándolos con tu tabla por `patient_id`

### Flujo recomendado

```
TCGA-BRCA
   ├── RNA-seq (Gene Expression Quantification)
   │       └─ patient_id → subtipo molecular / etiqueta clínica
   └── Diagnostic Slides (SVS)
           └─ patient_id → verificar correspondencia exacta con metadata GDC
                     └─ table: rna_file_id ↔ slide_file_id (mismo case/sample)
```

> **Advertencia:** No asumir que el SVS de un paciente corresponde al mismo espécimen
> del RNA-seq. Verificar siempre con `case_id` y `entity_submitter_id`.


## 17. Fuentes oficiales

| Recurso | URL |
|---|---|
| GDC Data Transfer Tool | https://gdc.cancer.gov/access-data/gdc-data-transfer-tool |
| GDC Data Portal | https://portal.gdc.cancer.gov/ |
| TCGA Sample Type Codes | https://gdc.cancer.gov/resources-tcga-users/tcga-code-tables/sample-type-codes |
| GDC API Documentation | https://docs.gdc.cancer.gov/API/Users_Guide/Getting_Started/ |
| GDC Data Access Processes | https://gdc.cancer.gov/access-data/data-access-processes-and-tools |
